In [1]:
import os
import sys
import session_info
import geopandas as gpd
import numpy as np
import scipy
import pandas as pd
pd.set_option('future.no_silent_downcasting', True)
import matplotlib.pyplot as plt
import mpl_toolkits
import rtree
from shapely.geometry import Polygon

"""
Reads previously compiled CDXXX census raw data files 
Clips all bordes to the Coastal US border 20m census file
Removes holes in each district
Calculates each district's border overlap with coastline and with state border
Calculates Polsby Popper
Outputs statistical shape data for each district
"""
Censuspath = "Census data/"
Shapepath = "Process data/"
Dataoutpath ="Process data/"

Projection = "ESRI:102008"
Precision = 0.01
#session_info.show()
print( "Libraries loaded")

Libraries loaded


In [2]:
def Border_clip( Geodf, Borderdf):
    """
    Takes a GeoPandas dataframe and clips its borders to the Borders in Borderdf
    Assumes GeoPandas dataframe has rows delineated by GEOID
    Cleans up any non polygon residuals
    Removes any remaining polygons with less than 100,000 sq meters in area
    """
    Districts_clipped = gpd.clip( Geodf, Borderdf, keep_geom_type=False).copy()
    Allgeoms = Districts_clipped.explode()
    Allpolys = Allgeoms[ Allgeoms.geometry.geom_type=="Polygon"].copy()
    Allpolys['geometry'] = Allpolys['geometry'].apply(lambda p: Polygon(p.exterior.coords)) ## make polygon out of exterior coordinates
    Allpolys['Area'] = Allpolys.area
    Allpolys = Allpolys[ Allpolys['Area'] >100000]
    DistrictsM = Allpolys.dissolve(by='GEOID', method='unary').reset_index()
    DistrictsM.drop(['Area'], axis=1, inplace=True)
    return( DistrictsM)

print( "Border_clip function defined")
    

Border_clip function defined


In [3]:
def Geo_check( Geodf ) :
    """
    Analyze the composition of the geo data frame
    Looks for differing geometry types
    """
    ### Count types of geometries found pre clipping
    geometry_types = {'Point': 0, 'LineString': 0, 'Polygon': 0, 'MultiPolygon': 0, 'GeometryCollection':0}

    for geom in Geodf['geometry']:
        geometry_types[geom.geom_type] += 1 

    print("Points:", geometry_types['Point'])
    print("Lines:", geometry_types['LineString'])
    print("Polygons:", geometry_types['Polygon'])
    print("MultiPolygons:", geometry_types['MultiPolygon'])
    print("GeometryCollections:", geometry_types['GeometryCollection'])
    print("Total Polys:", len( Geodf.explode()))
    print( "Total rows:", len(Geodf) )
    print( "Min area polygon:", int( min( Geodf.explode().area)))
    return
    
print( "Geo_check function defined")

Geo_check function defined


In [4]:
def Polsby(Geometry):
    """
    Calculates the Polsby Poppper compactness ratio of a given geometry. 
    Uses an area weighted average if more than one polygon.
    Uses only the exterior ring of each polgon
    Assumes coordinates have already been projected     
    Args: 
        Geometry (shapely geometry): A Shapely geometry object (e.g., Polygon, MultiPolygon). 
        
    Returns:
        float: The PolsbyPopper score
    """
    Geos = gpd.GeoDataFrame( geometry=[Geometry], crs=Projection)
    #Darea = Geos["geometry"].area
    Geos = Geos.explode( )
    #Geos['geometry'] = Geos['geometry'].apply(lambda p: Polygon(p.exterior.coords)) ## make polygon out of exterior coordinates
    Geos['Perimeter2'] = Geos["geometry"].length**2
    Geos["Area"] = Geos["geometry"].area
    Darea = Geos['Area'].sum()
    Geos["PolsbyI"] = (Geos["Area"] * 4 * 3.14159) /  Geos["Perimeter2"] 
    Geos["PolsbyW"] = Geos["PolsbyI"] * Geos["Area"] / Darea
    Polsby = Geos["PolsbyW"].sum() 
    
    return Polsby
print( "Polsby function defined")   

Polsby function defined


In [5]:
def Reock(Geometry):
    """
    Calculates the Reock compactness ratio of a given geometry. 
    Uses an area weighted average if more than one polygon.
    Assumes coordinates have already been projected and precison set    
    Args: 
        Geometry (shapely geometry): A Shapely geometry object (e.g., Polygon, MultiPolygon). 
        
    Returns:
        float: The Reock score
    """
    Geos = gpd.GeoDataFrame( geometry=[Geometry], crs=Projection)
    Geos = Geos.explode( )
    Geos['Bounding_circle_area'] = Geos["geometry"].minimum_bounding_circle().area
    Geos["Area"] = Geos["geometry"].area
    Darea = Geos['Area'].sum()
    Geos["ReockI"] = Geos["Area"] /  Geos["Bounding_circle_area"] 
    Geos["ReockW"] = Geos["ReockI"] * Geos["Area"] / Darea
    Reock = Geos["ReockW"].sum() 
    
    return Reock

def ReockX( District_geometry, State_geometry):
    """
    Calculates the Reock compactness ratio of a given geometry.
    Does not consider the area of minimum bounding circle outside of state lines
    Uses an area weighted average if more than one polygon.
    Assumes coordinates have already been projected and precison set    
    Args: 
        Geometry (shapely geometry): A Shapely geometry object (e.g., Polygon, MultiPolygon). 
        
    Returns:
        float: The Reock score
    """
    Geos = gpd.GeoDataFrame( geometry=[District_geometry], crs=Projection)
    Geos = Geos.explode( )
    Geos['Bounding_circle_area'] = Geos["geometry"].minimum_bounding_circle().intersection(State_geometry).area
    Geos["Area"] = Geos["geometry"].area
    Darea = Geos['Area'].sum()
    Geos["ReockI"] = Geos["Area"] /  Geos["Bounding_circle_area"] 
    Geos["ReockW"] = Geos["ReockI"] * Geos["Area"] / Darea
    ReockX = Geos["ReockW"].sum() 
    
    return ReockX

print( "Reock function defined")  

Reock function defined


In [6]:
def DSTborder( Geodf) :
    """
    Takes a geopandas dataframe with one states districts
    Assumes Projection is defined globally
    Dataframe has GEOID for each row
    Combines districts to form state
    Finds state border by difference from a buffered state border
    Finds intersection of districts border with state border
    Calcs length of district border that is also on state border DST
    Return dataframe with GEOID and DST
    """
    STunion = Geodf.geometry.union_all()        ### create state from union of all districts
    STall = gpd.GeoDataFrame( geometry=[STunion], crs=Projection) 
    STminus = STall.buffer(-1 )                   ### shrink starte by one meter all around    
    Border = STall.difference( STminus)            ### create one meter wide perimeter inside of state
    Border = gpd.GeoDataFrame(geometry=Border)
    DSTborder = gpd.overlay(Geodf, Border, how='intersection' )
    DSTborder["DSTperim"] =  DSTborder["geometry"].length/2 
    DSTborder['DSTperim'] = DSTborder['DSTperim'].fillna(0)
    DSTborder["DSTperim"] =  [int(x) for x in  DSTborder["DSTperim"]  ] 
    DSTborder.drop(['geometry'], axis=1, inplace=True)
    return( DSTborder[ ['GEOID', 'DSTperim'] ] )

def DSTborders( Districts):
    DSTdata = pd.DataFrame(columns=['GEOID', 'DSTperim'])
    States = Districts[ 'STATEFP'].unique()
    for STFP in States :
        DST = DSTborder( Districts[ Districts['STATEFP']==STFP ] )
        DSTdata = pd.concat( [DSTdata, DST] , ignore_index=True ) 
        DSTdata['DSTperim'] = DSTdata['DSTperim'].fillna(0)
    return( DSTdata )

print("DSTborder defined")

DSTborder defined


In [7]:
def CSTborder( Geodf, Coastdf ):
    """
    Takes a geopandas dataframe with districts
    Assumes all geopandas data frame have same Projection 
    Dataframe has GEOID for each row
    Finds intersection of districts with coastline
    Calcs length of district border that is also on coastline  CST
    Return dataframe with GEOID and CST
     """
    GeodfBuf = Geodf.copy()
    Coastal_Intersect = gpd.overlay( GeodfBuf, Coastdf, how='intersection', keep_geom_type=False )
    Coastal_Intersect['Coast_len'] = Coastal_Intersect.length
    Coastal_Districts = Coastal_Intersect.groupby( Coastal_Intersect["GEOID"]).agg( {'Coast_len':'sum'})
    Coastal_Districts['Coast_len'] = [ int(x) for x in Coastal_Districts['Coast_len'] ]
    Coastal_Districts['Coast_len'] = Coastal_Districts['Coast_len'].fillna(0)
    return( Coastal_Districts)

print("CSTborder defined")

CSTborder defined


In [8]:
##### Load US boundary file for clipping
File = 'cb_2023_us_nation_20m'
Shapedatafile = Censuspath + File + ".shp"
RawShapes = gpd.read_file( Shapedatafile)
USbox = Polygon( [ (-130,20),(-60,20), (-60,50), (-130,50) ] )        ### Contental US box
Shapes = gpd.clip( RawShapes, USbox)
Shapes.to_crs( Projection, inplace=True)
Shapes.geometry = Shapes.geometry.set_precision( 0.01 )
#Shapes.plot()
print( "Loaded ", File," with ", len( Shapes.explode()), " polygons")
#Shapes.head()
US48 = Shapes.copy()

Loaded  cb_2023_us_nation_20m  with  23  polygons


In [9]:
### Load coastline file for coastlline intersection calc
File = "tl_2023_us_coastline"
Shapedatafile = Censuspath + File + ".shp"
Coastline = gpd.read_file(Shapedatafile)
USbox = Polygon( [ (-130,20),(-60,20), (-60,50), (-130,50) ] )        ### Contental US box
Coastline = gpd.clip( Coastline, USbox)
Coastline.to_crs( crs=Projection, inplace=True )
#Coastline = gpd.clip( Coastline, Coastbox)
print(  File +" loaded ", len(Coastline)," rows of coastline data")
#Coastline.plot()
#Coastline.head()
#Geo_check( Coastline)

tl_2023_us_coastline loaded  3315  rows of coastline data


In [10]:
#### Calculate state metrics from district unions
def Calc_STdata( Districts ):
    STdata = pd.DataFrame(columns=['STATEFP', 'STarea', 'STperim','STpolsby','STreock','STgeometry'])
    States = Districts[ 'STATEFP'].unique()
    
    for STFP in States :
        STdists = Districts[ Districts['STATEFP'] == STFP]
        State = STdists.geometry.union_all()
        Sarea = int( State.area )
        Sperim = int( State.length )
        Spolsby =  Polsby(State)
        Sreock = Reock(State)
        STdata.loc[len(STdata)] = { 'STATEFP':STFP, 'STarea':Sarea, 'STperim':Sperim, 'STpolsby':Spolsby, 'STreock':Sreock, 'STgeometry':State }

    return( STdata )
#Districts = pd.merge( Districts, STdata, on='STATEFP', how='left')

print( 'Defined state data calc function')


Defined state data calc function


In [11]:
#### Calc district level metrics
#### Get district shape data 
File = 'CD115raw'              ### For mid decadde analysis
Fout = 'CD115_stats'
Shapedatafile = Shapepath + File + ".shp"
Dataoutfile = Dataoutpath + Fout + ".csv"

Districts = gpd.read_file( Shapedatafile)
print(  File +" loaded ", len(Districts)," rows of data")
Districts =  Border_clip( Districts, US48)
print('Clipped borders')

STATE_data = Calc_STdata( Districts )
Districts = pd.merge( Districts, STATE_data, on='STATEFP', how='left')
print('State calcs done')

Districts['Dperimeter'] = Districts.length
Districts['Dperimeter'] = [int(x) for x in Districts['Dperimeter'] ] 
Districts['Darea'] = Districts.area
Districts['Darea'] = [int(x) for x in Districts['Darea'] ] 
Coast = CSTborder(Districts, Coastline)
Districts = pd.merge( Districts, Coast, on='GEOID', how='left')
Districts['Coast_len'] = Districts['Coast_len'].fillna(0)
Districts['PolsbyW'] = Districts.geometry.apply( Polsby )
Districts['Reock'] = Districts.geometry.apply( Reock )
#Districts['ReockX'] = Districts.apply( lambda row: ReockX(row['geometry'], row['STgeometry']) )
Districts['ReockX'] = Districts.apply(lambda row: ReockX(row.geometry, row.STgeometry), axis=1 )
print( 'District calcs done')

District_ST_border = DSTborders( Districts)
Districts = pd.merge( Districts, District_ST_border, on='GEOID', how='left')
Districts['DSTperim'] = Districts['DSTperim'].fillna(0) 
print( 'DST calc done' )
#Districts.head()

CD115raw loaded  436  rows of data
Clipped borders
State calcs done
District calcs done
DST calc done


In [12]:
Data_columns = [ 'GEOID', 'STATEFP', 'DISTRICT', 'SESSN', 'Dperimeter', 'Darea', 'PolsbyW','Reock','ReockX', 'DSTperim','Coast_len', 'STarea', 'STperim', 'STpolsby','STreock']
Data_out = Districts[ Data_columns ]
Data_out.to_csv( Dataoutfile, index=False )
print( 'Done writing ', len( Data_out ), 'rows to ', Dataoutfile)
Data_out.head()

Done writing  436 rows to  Process data/CD115_stats.csv


,GEOID,STATEFP,DISTRICT,SESSN,Dperimeter,Darea,PolsbyW,Reock,ReockX,DSTperim,Coast_len,STarea,STperim,STpolsby,STreock
0,0101,01,01,115,1236875,16168059507,0.132805,0.439890,0.658022,552625,152078.0,133946536477,1860717,0.486161,0.489656
1,0102,01,02,115,1212463,26573517449,0.227155,0.494874,0.619276,300871,0.0,133946536477,1860717,0.486161,0.489656
2,0103,01,03,115,1072199,20009532822,0.218723,0.334680,0.580080,315423,0.0,133946536477,1860717,0.486161,0.489656
3,0104,01,04,115,1284565,23595746485,0.179693,0.374434,0.438437,220405,0.0,133946536477,1860717,0.486161,0.489656
4,0105,01,05,115,649469,9985457375,0.297482,0.246844,0.463631,256997,0.0,133946536477,1860717,0.486161,0.489656
